In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input/datasets/kadf.head()kulthisside/quote-dataset-csv"):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/kakulthisside/quote-dataset-csv/qoute_dataset.csv")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
quotes = df['quote']
quotes.head()

In [ ]:
quotes = quotes.str.lower()

In [ ]:
import string
translator = str.maketrans('', '', string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator))

In [ ]:
quotes[0]

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [ ]:
vocab_size = 8978
tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(quotes)

In [ ]:
word_index = tokenizer.word_index
print(len(word_index))
list(word_index.items())[:10]

In [ ]:
sequence = tokenizer.texts_to_sequences(quotes)

In [ ]:
for i in range(5):
    print(quotes[i])

In [ ]:
for i in range(5):
    print(sequence[i])

In [ ]:
X = []
y = []
for seq in sequence:
    for i in range(1, len(seq)):
        input_seq = seq[:i]
        output_seq = seq[i]
        X.append(input_seq)
        y.append(output_seq)

In [ ]:
len(X)

In [ ]:
len(y)

In [ ]:
max_len = max(len(x) for x in X)
print(max_len)

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
x_padded = pad_sequences(X, maxlen=max_len, padding='pre')

In [ ]:
x_padded[0]

In [ ]:
y = np.array(y)

In [ ]:
x_padded.shape

In [ ]:
y.shape

In [ ]:
from tensorflow.keras.utils import to_categorical
y_one_hot = to_categorical(y, num_classes=vocab_size)

In [ ]:
y_one_hot.shape

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, LSTM, Dense

In [ ]:
embedding_dim = 50
rnn_units = 128

In [ ]:
rnn_model = Sequential()

rnn_model.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        input_length=max_len
    )
)

rnn_model.add(
    SimpleRNN(
        units=rnn_units,
        activation='tanh'
    )
)

rnn_model.add(
    Dense(vocab_size, activation='softmax')
)

In [ ]:
rnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
rnn_model.summary()

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense

lstm_model = Sequential([
    Input(shape=(max_len,)),
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim
    ),
    LSTM(rnn_units),
    Dense(vocab_size, activation='softmax')
])

lstm_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
lstm_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
lstm_model.summary()

In [ ]:
epochs = 100
batch_size = 128

In [ ]:
history_rnn = rnn_model.fit(
    x_padded,
    y,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.2
)

In [ ]:
history_lstm = lstm_model.fit(
    x_padded,
    y,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.2
)

In [ ]:
lstm_model.save("lstm_model.h5")

In [ ]:
from tensorflow.keras.models import load_model

lstm_model = load_model("lstm_model.h5")

In [ ]:
index_to_word = {}
for word, index in word_index.items():
  index_to_word[index] = word
     

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
def predictor(model,tokenizer,text,max_len):
  text = text.lower()

  seq = tokenizer.texts_to_sequences([text])[0]
  seq = pad_sequences([seq], maxlen=max_len, padding='pre')

  pred = model.predict(seq,verbose = 0)
  pred_index = np.argmax(pred)
  return index_to_word[pred_index]

In [ ]:
seed_text = "what are you"
next_word = predictor(lstm_model,tokenizer,seed_text,max_len)
print(next_word)

In [ ]:
def generate_text(model,tokenizer,seed_text,max_len,n_words):
  for _ in range(n_words):
    next_word = predictor(model,tokenizer,seed_text,max_len)
    if next_word == "":
      break
    seed_text += " " + next_word
  return seed_text

In [ ]:
seed = "are you a "
generate_text = generate_text(lstm_model,tokenizer,seed,max_len,10)
print(generate_text)

In [ ]:
import pickle
with open("tokenizer.pkl", "wb") as f:
  pickle.dump(tokenizer, f)

In [ ]:
with open("max_len.pkl", "wb") as f:
  pickle.dump(max_len, f)
     